In [30]:
import pandas as pd
import json
import numpy as np

results_df = pd.read_csv('../../Our Datasets/classifier_results_logreg.csv')

possible_match_ids = [1886347, 1899585]

match_id = possible_match_ids[0]

results_df = results_df[results_df['match_id'] == match_id]

results_df.drop(columns=['match_id', 'row_index', 'Unique ID', 'event_index', 'source_file', 'event_row_index', 'rec_team_id', 'true_value'], inplace=True)

results_df

,predicted_class,predicted_probability,frame_start,frame_end,rec_player_id,rec_team_short,dist_to_attacking_goal,team_out_of_possession_phase_type,player_targeted_dangerous,is_drawing,is_middle_third
0,0,0.196989,387.0,430.0,735578.0,Newcastle,99.575170,high_block,0.0,1,0
1,1,0.818607,1492.0,1492.0,735574.0,Newcastle,34.427403,chaotic/disruption,0.0,1,0
2,1,0.524581,3799.0,3799.0,50978.0,Newcastle,51.188596,chaotic/disruption,1.0,1,1
3,1,0.553058,3854.0,3866.0,38673.0,Auckland FC,47.054798,chaotic/disruption,1.0,1,1
4,1,0.555331,4452.0,4452.0,795507.0,Newcastle,72.213891,medium_block,1.0,1,1
5,1,0.504472,7480.0,7480.0,735573.0,Newcastle,53.900933,chaotic/disruption,0.0,1,1
6,1,0.874515,8011.0,8011.0,51649.0,Newcastle,18.581808,chaotic/disruption,0.0,1,0
7,1,0.866691,9289.0,9301.0,23418.0,Auckland FC,30.829612,low_block,0.0,1,0
8,0,0.323647,9686.0,9696.0,50978.0,Newcastle,75.306988,high_block,0.0,1,0
9,0,0.338456,10163.0,10170.0,795507.0,Newcastle,72.861567,high_block,0.0,1,0


In [31]:
def time_to_seconds(time_str):
    if time_str is None:
        return 90 * 60  # 120 minutes = 7200 seconds
    h, m, s = map(int, time_str.split(':'))
    return h * 3600 + m * 60 + s

file_path = f"../../data/matches/{match_id}/{match_id}_match.json"

with open(file_path, "r") as f:
    raw_match_data = json.load(f)

# The output has nested json elements. We process them
raw_match_df = pd.json_normalize(raw_match_data, max_level=2)
raw_match_df["home_team_side"] = raw_match_df["home_team_side"].astype(str)

players_df = pd.json_normalize(
    raw_match_df.to_dict("records"),
    record_path="players",
    meta=[
        "home_team_score",
        "away_team_score",
        "date_time",
        "home_team_side",
        "home_team.name",
        "home_team.id",
        "away_team.name",
        "away_team.id",
    ],  # data we keep
)


# Take only players who played and create their total time
players_df = players_df[
    ~((players_df.start_time.isna()) & (players_df.end_time.isna()))
]

# Create a flag for GK
players_df["is_gk"] = players_df["player_role.acronym"] == "GK"

# Add a flag if the given player is home or away
players_df["match_name"] = (
    players_df["home_team.name"] + " vs " + players_df["away_team.name"]
)


# Add a flag if the given player is home or away
players_df["home_away_player"] = np.where(
    players_df.team_id == players_df["home_team.id"], "Home", "Away"
)

# Create flag from player
players_df["team_name"] = np.where(
    players_df.team_id == players_df["home_team.id"],
    players_df["home_team.name"],
    players_df["away_team.name"],
)

# Figure out sides
players_df[["home_team_side_1st_half", "home_team_side_2nd_half"]] = (
    players_df["home_team_side"]
    .astype(str)
    .str.strip("[]")
    .str.replace("'", "")
    .str.split(", ", expand=True)
)
# Clean up sides
players_df["direction_player_1st_half"] = np.where(
    players_df.home_away_player == "Home",
    players_df.home_team_side_1st_half,
    players_df.home_team_side_2nd_half,
)
players_df["direction_player_2nd_half"] = np.where(
    players_df.home_away_player == "Home",
    players_df.home_team_side_2nd_half,
    players_df.home_team_side_1st_half,
)


# Clean up and keep the columns that we want to keep about

columns_to_keep = [
    "id",
    "short_name",
    "team_name",
    "player_role.name",
]
players_df = players_df[columns_to_keep]

players_df

,id,short_name,team_name,player_role.name
0,38673,G. May,Auckland FC,Center Forward
1,51713,C. Elliott,Auckland FC,Right Back
2,50951,J. Brimmer,Auckland FC,Center Forward
3,50978,C. Timmins,Newcastle United Jets FC,Left Defensive Midfield
4,133498,F. De Vries,Auckland FC,Left Back
5,33697,N. Pijnaker,Auckland FC,Left Center Back
6,51667,D. Hall,Auckland FC,Right Center Back
7,14736,L. Verstraete,Auckland FC,Defensive Midfield
11,735573,T. Aquilina,Newcastle United Jets FC,Left Winger
12,51009,R. Scott,Newcastle United Jets FC,Goalkeeper


In [32]:
summary_df = pd.merge(results_df, players_df, left_on='rec_player_id', right_on='id', how='right')

summary_df

,predicted_class,predicted_probability,frame_start,frame_end,rec_player_id,rec_team_short,dist_to_attacking_goal,team_out_of_possession_phase_type,player_targeted_dangerous,is_drawing,is_middle_third,id,short_name,team_name,player_role.name
0,1.0,0.553058,3854.0,3866.0,38673.0,Auckland FC,47.054798,chaotic/disruption,1.0,1.0,1.0,38673,G. May,Auckland FC,Center Forward
1,1.0,0.723137,19986.0,19986.0,38673.0,Auckland FC,45.116494,medium_block,1.0,1.0,1.0,38673,G. May,Auckland FC,Center Forward
2,1.0,0.964283,47322.0,47322.0,38673.0,Auckland FC,39.455101,transition/quick_break,1.0,1.0,0.0,38673,G. May,Auckland FC,Center Forward
3,0.0,0.187783,7447.0,7456.0,38673.0,Auckland FC,49.538398,chaotic/disruption,0.0,1.0,1.0,38673,G. May,Auckland FC,Center Forward
4,0.0,0.117465,27584.0,27598.0,38673.0,Auckland FC,69.013071,chaotic/disruption,0.0,1.0,1.0,38673,G. May,Auckland FC,Center Forward
5,0.0,0.433564,4437.0,4437.0,51713.0,Auckland FC,40.190030,chaotic/disruption,0.0,1.0,0.0,51713,C. Elliott,Auckland FC,Right Back
6,0.0,0.090006,1667.0,1715.0,50951.0,Auckland FC,74.534154,high_block,0.0,1.0,0.0,50951,J. Brimmer,Auckland FC,Center Forward
7,0.0,0.466480,41626.0,41626.0,50951.0,Auckland FC,35.128081,chaotic/disruption,0.0,1.0,0.0,50951,J. Brimmer,Auckland FC,Center Forward
8,1.0,0.524581,3799.0,3799.0,50978.0,Newcastle,51.188596,chaotic/disruption,1.0,1.0,1.0,50978,C. Timmins,Newcastle United Jets FC,Left Defensive Midfield
9,0.0,0.323647,9686.0,9696.0,50978.0,Newcastle,75.306988,high_block,0.0,1.0,0.0,50978,C. Timmins,Newcastle United Jets FC,Left Defensive Midfield


In [38]:
player_df_recoveries = (
    summary_df
    .groupby(['short_name', 'rec_team_short'])
    .agg(
        recoveries_count = ('predicted_class', 'count'),
        predicted_class_count = ('predicted_class', 'sum'),
        avg_pred_prob = ('predicted_probability', 'mean')
    )
    .reset_index()
)

player_df_recoveries.sort_values(by='predicted_class_count', ascending=False, inplace=True)

player_df_recoveries

,short_name,rec_team_short,recoveries_count,predicted_class_count,avg_pred_prob
11,G. May,Auckland FC,5,3.0,0.509145
17,L. Verstraete,Auckland FC,3,2.0,0.682284
13,K. Grozos,Newcastle,2,2.0,0.777862
14,L. Bayliss,Newcastle,5,2.0,0.361199
23,T. Aquilina,Newcastle,4,2.0,0.387704
0,A. Paulsen,Auckland FC,3,1.0,0.424769
2,B. Gibson,Newcastle,2,1.0,0.470199
1,A. Å uÅ¡njar,Newcastle,2,1.0,0.526196
21,P. Cancar,Newcastle,3,1.0,0.475429
7,D. Ingham,Newcastle,1,1.0,0.667639
